# Lab 13 — Custom AI Agent: Test Coverage Gap Report

## Overview

In this lab you will build and run a **custom LangChain agent** that automatically:

1. Retrieves all Jira stories from the RAG vector store
2. Finds matching TestRail test cases for each story
3. Classifies coverage: **Covered** / **Partial** / **No Coverage**
4. Exports a colour-coded **Excel traceability matrix** with a summary dashboard

### Architecture

```
User Prompt
    │
    ▼
LangChain Agent (GPT-4o)
    │
    ├── Tool 1: rag_retriever(query, top_k)
    │           └── FAISS vector store (Jira + TestRail + API docs)
    │
    └── Tool 2: export_gap_report_to_excel(coverage_data_json)
                └── Styled .xlsx with 2 sheets
```

### Files

| File | Purpose |
|---|---|
| `custom-agent/coverage_gap_agent.py` | Main agent script |
| `custom-agent/requirements.txt` | Python dependencies |
| `custom-agent/reports/` | Output Excel reports |
| `rag/.env` | Shared API keys (OpenAI, Jira, TestRail) |

## Prerequisites

Before running this lab, make sure:

- ✅ Lab 12 is complete — FAISS index is built (`rag/faiss_index/` exists)
- ✅ `rag/.env` has a valid `OPENAI_API_KEY`
- ✅ Python 3.11+ is installed (we use the global Python313 installation)

### Verify the FAISS index exists

Run the cell below to confirm the vector store is ready:

In [ ]:
import os

faiss_path = os.path.join("..", "..", "rag", "faiss_index")
if os.path.exists(faiss_path):
    files = os.listdir(faiss_path)
    print(f"✅ FAISS index found: {files}")
else:
    print("❌ FAISS index NOT found. Run Lab 12 first (python rag/main.py)")

## Step 1 — Install Dependencies

Install required packages into the global Python environment.

> **Note:** We use the global Python313 installation because the .venv in this project does not have pip.exe. If you are using a different environment, adjust the path accordingly.

In [ ]:
! pip install -r ../../custom-agent/requirements.txt

### requirements.txt contents

```
langchain
langchain-openai
langchain-community
openpyxl
requests
python-dotenv
faiss-cpu
openai
```

## Step 2 — Understand the Agent Code

Open `custom-agent/coverage_gap_agent.py` and review:

### Tool 1: `rag_retriever`

```python
@tool
def rag_retriever(query: str, top_k: int = 10) -> str:
    """Query the RAG vector store to retrieve relevant indexed documents."""
    from get_relevant_docs import get_relevant_docs
    docs = get_relevant_docs(query, top_k=top_k)
    return json.dumps(docs, indent=2)
```

This tool wraps the same FAISS vector store used in Lab 12. The agent calls it with:
- A broad query to fetch **all Jira stories** (`top_k=30`)
- A focused query per story to find **matching TestRail cases** (`top_k=8`)

### Tool 2: `export_gap_report_to_excel`

```python
@tool
def export_gap_report_to_excel(coverage_data_json: str) -> str:
    """Export the test coverage gap analysis to a styled Excel workbook."""
    # Writes to custom-agent/reports/coverage_gap_report_YYYYMMDD_HHMMSS.xlsx
```

Produces an Excel file with **2 sheets**:
- **Sheet 1 — Coverage Gap Report**: Colour-coded traceability matrix
  - 🟢 Green = Covered (≥2 test cases)
  - 🟡 Yellow = Partial (1 test case)
  - 🔴 Red = No Coverage
- **Sheet 2 — Summary**: Dashboard with counts and gap story list

### System Prompt (Agent Instructions)

The agent follows 6 steps:
1. Retrieve all Jira stories via RAG (`top_k=30`)
2. Per story → retrieve matching TestRail cases (`top_k=8`)
3. Classify: Covered / Partial / No Coverage
4. Build a JSON array of results
5. **Call `export_gap_report_to_excel`** with the JSON
6. Report the file path + summary statistics

## Step 3 — Run the Agent

Run the agent from the `custom-agent/` directory using global Python313.

> ⏱️ **Expected runtime:** 60–120 seconds (the agent makes multiple RAG + GPT-4o calls)

Open Terminal and run below command:



In [ ]:

cd ../../custom-agent
python coverage_gap_agent.py


### Expected Output

```
=================================================================
  Test Coverage Gap Report Agent
=================================================================

=================================================================
AGENT OUTPUT:
=================================================================
### Test Coverage Gap Report Summary
- **Exact File Path**: `C:\...\custom-agent\reports\coverage_gap_report_20260310_125837.xlsx`
- **Total Stories Analyzed**: 19
- **Covered Count**: 2
- **Partial Coverage Count**: 4
- **No Coverage Count**: 13

### Stories with No Coverage
- **KAN-5**: EC-E1 Product Catalog
- **KAN-10**: EC-E6 UI/UX & Navigation
- ...
```

## Step 4 — Verify the Excel Report

Check that the Excel file was created in `custom-agent/reports/`:

## Step 5 — Inspect the Excel Output

Open the generated `.xlsx` file to see:

### Sheet 1: Coverage Gap Report

| Jira Key | Summary | Issue Type | Status | Test Cases | Coverage |
|---|---|---|---|---|---|
| KAN-14 | Add to Cart | Story | Done | C59, C61 | 🟢 Covered |
| KAN-6 | Login | Story | Done | C58 | 🟡 Partial |
| KAN-5 | Product Catalog | Epic | In Progress | — | 🔴 No Coverage |

**Colour legend:**
- 🟢 **Green** — Covered: 2 or more TestRail cases found
- 🟡 **Yellow** — Partial: exactly 1 TestRail case found
- 🔴 **Red** — No Coverage: 0 TestRail cases found

### Sheet 2: Summary Dashboard

| Metric | Count | Percentage |
|---|---|---|
| Total Jira Stories | 19 | 100% |
| Covered | 2 | 10.5% |
| Partial Coverage | 4 | 21.1% |
| No Coverage (Gap) | 13 | 68.4% |

## How It Works — Agent Flow

```
1. User sends prompt:
   "Generate a complete test coverage gap report for the KAN project."

2. Agent calls rag_retriever(query="all Jira KAN project stories", top_k=30)
   → Returns 19 Jira issues (KAN-1 through KAN-20)

3. For each story, agent calls rag_retriever(query="<story summary>", top_k=8)
   → Filters docs where id starts with "testrail_"
   → Extracts test case IDs: C59, C61, etc.

4. Agent classifies each story:
   - KAN-14 → C59, C61 → Covered
   - KAN-6  → C58      → Partial
   - KAN-5  → (none)   → No Coverage

5. Agent calls export_gap_report_to_excel(json_array)
   → Writes coverage_gap_report_YYYYMMDD_HHMMSS.xlsx

6. Agent returns file path + summary statistics
```

## Key Concepts Demonstrated

| Concept | Implementation |
|---|---|
| **Custom LangChain Agent** | `create_agent(model, tools, system_prompt)` |
| **RAG as a Tool** | `rag_retriever` wraps `get_relevant_docs()` from Lab 12 |
| **Action Tool** | `export_gap_report_to_excel` performs a real action (writes file) |
| **System Prompt Engineering** | Step-by-step instructions force deterministic agent behaviour |
| **Tool Calling** | Agent decides when and how to call each tool based on the prompt |
| **Traceability** | Jira ↔ TestRail cross-reference without direct API calls |

## Troubleshooting

| Problem | Solution |
|---|---|
| `ModuleNotFoundError: langchain` | Run Step 1 to install dependencies |
| `FAISS index not found` | Build the index: `cd rag && python main.py` |
| Agent returns 0 stories | FAISS path issue — check `rag/get_relevant_docs.py` uses absolute path |
| Excel file not created | Agent described export instead of calling it — check system prompt enforcement |
| `openai.AuthenticationError` | Check `OPENAI_API_KEY` in `rag/.env` |
| Script exits with no output | Run with `python coverage_gap_agent.py` from `custom-agent/` directory |